# **<font color="Gray"> Investment Research Agent: An Agentic AI Financial Analysis System — Team 2 </font>**

## **<font color="green">Heads Up, 429 or provider overload errors don’t necessarily mean the code is wrong, it could be due to reaching the limits of the model or model being temporarily down. The code automatically tries fallback models. If all attempts fail, wait and retry the failed cell. If the problem persists, try another available free model from OpenRouter.</font>**

---
# 1. Environment Setup and Dependencies

In [1]:
!pip install -q openai yfinance jsonschema

In [2]:
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
import json
import re
import time
import uuid
from getpass import getpass

import pandas as pd
from jsonschema import Draft202012Validator, ValidationError
from openai import OpenAI, APIStatusError, APIConnectionError
import yfinance as yf

PLANNING_MODELS = (
    "nvidia/nemotron-3-super-120b-a12b:free",
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "qwen/qwen3.8-27b:free",
)

RESEARCH_MODELS = (
    "nvidia/nemotron-3-super-120b-a12b:free",
    "qwen/qwen3.8-27b:free",
    "openrouter/free",
)

PERIODS = ("1mo", "3mo", "6mo", "1y", "2y", "5y")
TOOL_NAMES = ("get_stock_prices", "get_company_financials")


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def normalize_ticker(ticker):
    if not isinstance(ticker, str) or not ticker.strip():
        raise ValueError("Provide a nonempty stock symbol.")
    return ticker.strip().upper()


def make_task(ticker, question, price_period=None, excluded_tools=()):
    """Optional structured constraints supplement, rather than replace, the question."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Provide a nonempty research question.")
    if price_period is not None and price_period not in PERIODS:
        raise ValueError(f"price_period must be one of {PERIODS}.")
    if isinstance(excluded_tools, str) or set(excluded_tools) - set(TOOL_NAMES):
        raise ValueError("excluded_tools must contain registered tool names.")
    return {
        "run_id": str(uuid.uuid4()),
        "ticker": normalize_ticker(ticker),
        "question": question.strip(),
        "price_period": price_period,
        "excluded_tools": list(excluded_tools),
    }


def safe_error(error):
    """Keep provider errors useful without persisting credentials or long payloads."""
    return re.sub(r"sk-[A-Za-z0-9_-]+", "[REDACTED]", str(error))[:300]

In [3]:
# Each teammate enters their own key. Never put the key in the notebook source.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getpass("Enter your OpenRouter API key: ").strip(),
    max_retries=0,
    timeout=60.0,
)
print("OpenRouter client configured.")

Enter your OpenRouter API key: ··········
OpenRouter client configured.


## 2. Base Agent

Following the lab 7, each agent has a name, role, model choices, session memory, process, and send_to. LLM requests have at most three free-model attempts. Invalid outputs are rejected; unsuccessful requests remain errors.

In [4]:
class Agent:
    """Base class: identity, role, LLM calls, memory and send_to."""

    def __init__(self, name, role, client, models, event_log=None, verbose=True, retry_delay=2):
        if not models or len(models) > 3 or any(
            m != "openrouter/free" and not m.endswith(":free") for m in models
        ):
            raise ValueError("Configure one to three free model IDs only.")
        self.name, self.role = name, role
        self.client, self.models = client, tuple(dict.fromkeys(models))
        self.memory = []  # Session history only: NOT learning across notebook restarts.
        self.event_log = event_log if event_log is not None else []
        self.verbose, self.retry_delay = verbose, retry_delay
        self.last_attempts = []

    def _log(self, message):
        if self.verbose:
            print(message)

    def call_llm(self, prompt, validator, response_format=None):
        """Bounded fallback, including invalid outputs; never fabricate a response."""
        self.last_attempts = []
        messages = [
            {"role": "system", "content": f"You are {self.name}. {self.role}"},
            {"role": "user", "content": prompt},
        ]
        for index, model in enumerate(self.models):
            if index and self.retry_delay:
                time.sleep(self.retry_delay)
            self._log(f"{self.name}: attempt {index + 1}/{len(self.models)} — {model}")
            attempt = {"requested_model": model, "status": "failed"}
            self.last_attempts.append(attempt)
            request = dict(model=model, messages=messages, max_tokens=4096, temperature=1.0)
            if response_format:
                request.update(response_format=response_format,
                               extra_body={"provider": {"require_parameters": True}})
            try:
                response = self.client.with_options(max_retries=0).chat.completions.create(**request)
            except APIStatusError as error:
                attempt["error"] = f"HTTP {error.status_code}"
                # Invalid credentials, billing and malformed requests need a user/code fix.
                if error.status_code not in {404, 408, 429, 500, 502, 503, 504}:
                    raise
                reason = attempt["error"]
            except APIConnectionError:
                reason = "Connection failed or timed out"
            else:
                attempt["actual_model"] = getattr(response, "model", None)
                provider_error = getattr(response, "error", None)
                if provider_error:
                    reason = safe_error(provider_error.get("message", "Provider error"))
                    if str(provider_error.get("code")) in {"400", "401", "402", "403", "413", "422"}:
                        attempt["error"] = reason
                        raise RuntimeError(reason)
                elif not getattr(response, "choices", None):
                    reason = "No choices returned"
                else:
                    choice = response.choices[0]
                    content = choice.message.content
                    if choice.finish_reason == "content_filter":
                        attempt["error"] = "Provider filtered the response"
                        raise RuntimeError(attempt["error"])
                    if choice.finish_reason != "stop":
                        reason = f"Incomplete response: {choice.finish_reason}"
                    elif not isinstance(content, str) or not content.strip():
                        reason = "Empty response"
                    else:
                        try:
                            value = validator(content)
                        except (ValueError, ValidationError) as error:
                            reason = "Rejected output: " + safe_error(error)
                        else:
                            attempt["status"] = "success"
                            self._log(f"{self.name}: accepted response from {response.model}")
                            return value, response.model, deepcopy(self.last_attempts)
            attempt["error"] = reason
            self._log(f"{self.name}: {reason}")
            # Corrective feedback accompanies the next bounded fallback attempt.
            messages.append({"role": "user", "content":
                             "Previous attempt failed validation or availability. "
                             f"Return a complete answer to the original task. Detail: {reason}"})
        raise RuntimeError(f"{self.name}: all {len(self.models)} free attempts failed. "
                           "No successful result was fabricated. Try later.")

    def process(self, task):
        raise NotImplementedError("Each specialist implements its own process method.")

    def remember(self, task, result):
        self.memory.append({"run_id": task["run_id"], "task": task["question"],
                            "result_status": result["status"], "recorded_at": utc_now()})

    def send_to(self, other_agent, task):
        event = {"run_id": task["run_id"], "from": self.name, "to": other_agent.name,
                 "task": task["question"], "status": "started", "started_at": utc_now()}
        self.event_log.append(event)
        self._log(f"{self.name} → {other_agent.name}: {task['question']}")
        try:
            result = other_agent.process(deepcopy(task))
        except Exception as error:
            event.update(status="failed", error=safe_error(error), finished_at=utc_now())
            raise
        event.update(status=result.get("status", "completed"), finished_at=utc_now())
        return result

---
# 3. Financial Data Retrieval and Tools

Two tools retrieve adjusted daily prices and up to four quarterly income statements. Each result includes its source, retrieval time and status. Four quarters support quarter-over-quarter comparisons; earnings-announcement dates are not retrieved.

In [5]:
class YahooFinanceTools:
    """Yahoo Finance retrieval tools shared by the research agent."""

    def __init__(self, ticker_factory=None):
        self.ticker_factory = ticker_factory or yf.Ticker
        self.functions = {
            "get_stock_prices": self.get_stock_prices,
            "get_company_financials": self.get_company_financials,
        }

    def _result(self, ticker, period):
        return {
            "ticker": normalize_ticker(ticker), "period": period,
            "source": "Yahoo Finance via yfinance", "retrieved_at": utc_now(),
            "status": "error", "data": [],
        }

    def get_stock_prices(self, ticker, period="1mo"):
        """Daily adjusted OHLC, volume and corporate-action columns if available."""
        result = self._result(ticker, period)
        try:
            if period not in PERIODS:
                raise ValueError("Unsupported price-history period.")
            frame = self.ticker_factory(result["ticker"]).history(
                period=period, interval="1d", auto_adjust=True, actions=True
            )
            if frame.empty or "Close" not in frame or frame["Close"].notna().sum() == 0:
                raise ValueError("No usable closing-price data returned.")
            frame = frame.sort_index().copy()
            frame.index.name = "Date"
            frame = frame.reset_index()
            frame["Date"] = frame["Date"].astype(str)
            result.update(
                status="success", adjustment="auto_adjust=True",
                row_count=len(frame),
                data=json.loads(frame.to_json(orient="records", date_format="iso")),
            )
        except Exception as error:
            result["error"] = safe_error(error)
        return result

    def get_company_financials(self, ticker):
        """Up to four reported quarters, newest first; values retain Yahoo units."""
        result = self._result(ticker, "Up to four reported quarters")
        try:
            frame = self.ticker_factory(result["ticker"]).quarterly_income_stmt
            if frame.empty:
                raise ValueError("No quarterly income statements returned.")
            frame = frame.T.sort_index(ascending=False).head(4).copy()
            if not any(c in frame and frame[c].notna().any()
                       for c in ("Total Revenue", "Net Income")):
                raise ValueError("Revenue and net income are missing from these statements.")
            frame.index.name = "quarter_end"
            frame = frame.reset_index()
            frame["quarter_end"] = frame["quarter_end"].astype(str)
            result.update(
                status="success", row_count=len(frame),
                units="Original reporting units; currency not independently verified",
                data=json.loads(frame.to_json(orient="records", date_format="iso")),
            )
        except Exception as error:
            result["error"] = safe_error(error)
        return result


TOOL_DESCRIPTIONS = {
    "get_stock_prices": "Adjusted daily prices, volume, and available dividends/splits.",
    "get_company_financials": "Up to four reported quarterly income statements.",
}

CALL_SCHEMA = {
    "type": "object",
    "properties": {
        "calls": {
            "type": "array", "minItems": 1, "maxItems": 2,
            "items": {"anyOf": [
                {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "enum": ["get_stock_prices"]},
                        "arguments": {
                            "type": "object",
                            "properties": {
                                "ticker": {"type": "string", "minLength": 1},
                                "period": {"type": "string", "enum": list(PERIODS)},
                            },
                            "required": ["ticker", "period"], "additionalProperties": False,
                        },
                    },
                    "required": ["name", "arguments"], "additionalProperties": False,
                },
                {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "enum": ["get_company_financials"]},
                        "arguments": {
                            "type": "object",
                            "properties": {"ticker": {"type": "string", "minLength": 1}},
                            "required": ["ticker"], "additionalProperties": False,
                        },
                    },
                    "required": ["name", "arguments"], "additionalProperties": False,
                },
            ]},
        },
    },
    "required": ["calls"], "additionalProperties": False,
}
CALL_VALIDATOR = Draft202012Validator(CALL_SCHEMA)

---
# 4. Research Planning and Dynamic Tool Use

Coordinator creates the research plan and delegates the retrieval task. ResearcherAgent selects and executes the relevant tools. The coordinator plans the request and sends it to the researcher; the researcher returns evidence to the coordinator. These are executed send_to calls, recorded in the delegation log.

In [6]:
# Researcher (Yahoo data retrieval) Agent
class ResearcherAgent(Agent):
    def __init__(self, client, tools, **kwargs):
        role = (
            "You are the financial evidence collection agent. Select relevant tools using "
            "the original question, assisted by the plan. The original question and explicit "
            "constraints take priority. Never retrieve explicitly excluded information. "
            "For prices/volume select get_stock_prices; for quarterly revenue/profit select "
            "get_company_financials. Prices-only requests require only get_stock_prices. "
            "Match the ticker and requested period. Select all needed retrieval tools, "
            "at most once each. Return only the requested JSON; do not invent financial results. "
            "Tools: " + json.dumps(TOOL_DESCRIPTIONS)
        )
        super().__init__("Researcher", role, client, RESEARCH_MODELS, **kwargs)
        self.tools = tools

    def validate_selection(self, text, task):
        selection = json.loads(text)
        CALL_VALIDATOR.validate(selection)
        seen = set()
        # Validate the ENTIRE batch before any tool can run.
        for call in selection["calls"]:
            name, args = call["name"], call["arguments"]
            if name not in self.tools.functions or name in task["excluded_tools"]:
                raise ValueError(f"Tool is unavailable or explicitly excluded: {name}")
            if name in seen:
                raise ValueError("Select each tool at most once.")
            seen.add(name)
            if normalize_ticker(args["ticker"]) != task["ticker"]:
                raise ValueError("Tool ticker differs from the requested ticker.")
            args["ticker"] = task["ticker"]
            if (name == "get_stock_prices" and task["price_period"] is not None
                    and args["period"] != task["price_period"]):
                raise ValueError("The price period differs from the explicit request.")
        return selection

    def process(self, task):
        selection, model, attempts = self.call_llm(
            json.dumps(task), lambda text: self.validate_selection(text, task),
            response_format={"type": "json_schema", "json_schema": {
                "name": "research_tool_selection", "strict": True, "schema": CALL_SCHEMA}},
        )
        results = []
        for call in selection["calls"]:
            name, args = call["name"], call["arguments"]
            result = self.tools.functions[name](**args)
            results.append({"tool": name, "arguments": args, "result": result})
            self._log(f"{self.name}: {name}({args}) → {result['status']}")
        success_count = sum(r["result"]["status"] == "success" for r in results)
        status = ("success" if success_count == len(results)
                  else "partial" if success_count else "error")
        packet = {**task, "status": status, "tool_results": results,
                  "execution_model": model, "execution_attempts": attempts,
                  "selection_method": "LLM-selected JSON, validated before execution"}
        self.remember(task, packet)
        return packet

In [7]:
# Specialized Agents
class Coordinator(Agent):
    """Plans research, delegates retrieval, and receives the resulting evidence."""

    def __init__(self, client, **kwargs):
        role = (
            "You coordinate investment research and create its research plan. Write a numbered plan "
            "with 3–5 short actionable steps in plain language. Do not output tool-call "
            "tags, JSON or executable calls. Respect the original question and explicit "
            "tool exclusions. Describe relevant tools and their inputs, without claiming "
            "they have run. Use only these retrieval tools: " + json.dumps(TOOL_DESCRIPTIONS) +
            ". Valid price periods: " + ", ".join(PERIODS) + ". "
            "Three months is a calendar period, not exactly 90 days; trading data omit "
            "weekends and exchange holidays. Do not expand a requested period to fill those gaps. "
            "Four quarters permit quarter-over-quarter comparisons, not year-over-year. "
            "These tools do not retrieve earnings announcement dates, news, guidance, "
            "or analyst forecasts. Do not propose event reactions or invent unavailable facts. "
            "Mention relevant limitations. Numerical analysis is delegated to later specialists."
        )
        super().__init__("Coordinator", role, client, PLANNING_MODELS, **kwargs)
        self.team = {}
        self.last_result = None

    def plan_research(self, task):
        def validate_plan(text):
            if "<|" in text or not 3 <= len(re.findall(r"(?m)^\s*\d+[.)]\s", text)) <= 5:
                raise ValueError("Return a plain-language numbered plan with 3–5 steps.")
            return text.strip()

        plan, model, attempts = self.call_llm(json.dumps(task), validate_plan)
        result = {**task, "status": "planned", "plan": plan,
                  "planning_model": model, "planning_attempts": attempts}
        return result

    def add_agent(self, agent):
        if agent.name in self.team:
            raise ValueError(f"Duplicate team member: {agent.name}")
        self.team[agent.name] = agent

    def process(self, packet):
        """Receive a completed evidence packet from Researcher.send_to()."""
        if "tool_results" not in packet:
            raise ValueError("Coordinator expected an evidence packet.")
        self.remember(packet, packet)
        return packet

    def delegate_project(self, task):
        self.last_result = None
        researcher = self.team["Researcher"]
        plan = self.plan_research(task)
        evidence = self.send_to(researcher, plan)
        received = researcher.send_to(self, evidence)
        received["delegation_log"] = deepcopy([
            event for event in self.event_log if event["run_id"] == task["run_id"]
        ])
        self.last_result = received
        return received

# Multiagent System
class MultiAgentTeam:

    def __init__(self, client, tools=None, verbose=True, retry_delay=2):
        self.event_log = []
        self.tools = tools if tools is not None else YahooFinanceTools()
        options = dict(event_log=self.event_log, verbose=verbose, retry_delay=retry_delay)
        self.coordinator = Coordinator(client, **options)
        self.researcher = ResearcherAgent(client, self.tools, **options)
        self.coordinator.add_agent(self.researcher)

    def execute_project(self, ticker, question, price_period=None, excluded_tools=()):
        task = make_task(ticker, question, price_period, excluded_tools)
        return self.coordinator.delegate_project(task)

    def show_team_status(self):
        return pd.DataFrame([
            {"agent": a.name, "completed_tasks_this_session": len(a.memory)}
            for a in (self.coordinator, self.researcher)
        ])


def save_handoff(packet, path="research_handoff.json"):
    """Export evidence and delegation history, never clients or API credentials."""
    content = json.dumps(packet, indent=2, allow_nan=False)
    Path(path).write_text(content, encoding="utf-8")
    return str(Path(path).resolve())

In [8]:
# Construct agent objects
team = MultiAgentTeam(client)
print("Coordinator and Researcher created.")

Coordinator and Researcher created.


In [9]:
# AAPL Stock: planning, delegation and data collection
# The agents should select six months of prices and quarterly financial statements
# The result is an evidence packet for the next specialist, not a completed investment report.
research_run = team.execute_project(
    "AAPL",
    "How has the stock performed over the last six months, and how have "
    "revenue and net income changed across reported quarters?",
    price_period="6mo",
)

print("Research status:", research_run["status"])
print(research_run["plan"])
display(pd.DataFrame(research_run["delegation_log"])[["from", "to", "status"]])

for item in research_run["tool_results"]:
    print(item["tool"], item["arguments"], item["result"]["status"])
    if item["result"]["status"] == "success":
        table = pd.DataFrame(item["result"]["data"])
        if item["tool"] == "get_company_financials":
            columns = [c for c in ["quarter_end", "Total Revenue", "Net Income"] if c in table]
            table = table[columns]
        display(table.tail())
    else:
        print(item["result"].get("error", "Data unavailable"))

Coordinator: attempt 1/3 — nvidia/nemotron-3-super-120b-a12b:free
Coordinator: accepted response from nvidia/nemotron-3-super-120b-a12b:free
Coordinator → Researcher: How has the stock performed over the last six months, and how have revenue and net income changed across reported quarters?
Researcher: attempt 1/3 — nvidia/nemotron-3-super-120b-a12b:free
Researcher: accepted response from nvidia/nemotron-3-super-120b-a12b:free
Researcher: get_stock_prices({'ticker': 'AAPL', 'period': '6mo'}) → success
Researcher: get_company_financials({'ticker': 'AAPL'}) → success
Researcher → Coordinator: How has the stock performed over the last six months, and how have revenue and net income changed across reported quarters?
Research status: success
1. **Get six‑month price data** – Use the `get_stock_prices` tool with the ticker “AAPL” and the period “6mo” to obtain adjusted daily closing prices, volume, and any dividends/splits for the last six months.  

2. **Collect recent financial statements**

,from,to,status
0,Coordinator,Researcher,success
1,Researcher,Coordinator,success


get_stock_prices {'ticker': 'AAPL', 'period': '6mo'} success


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
121,2026-09-24 00:00:00-04:00,336.720001,338.910004,334.299988,335.920013,24733100,0.0,0.0
122,2026-09-25 00:00:00-04:00,336.040009,341.670013,334.529999,341.070007,30002500,0.0,0.0
123,2026-09-28 00:00:00-04:00,340.369995,342.989990,338.040009,338.399994,32820800,0.0,0.0
124,2026-09-29 00:00:00-04:00,336.970001,337.089996,328.700012,329.399994,38478000,0.0,0.0
125,2026-09-30 00:00:00-04:00,330.799988,339.500000,330.140015,333.019989,49931500,0.0,0.0


get_company_financials {'ticker': 'AAPL'} success


,quarter_end,Total Revenue,Net Income
0,2026-06-30,1.094170e+11,2.978900e+10
1,2026-03-31,1.111840e+11,2.957800e+10
2,2025-12-31,1.437560e+11,4.209700e+10
3,2025-09-30,1.024660e+11,2.746600e+10


In [10]:
# MSFT Stock: dynamic tool-selection check
# The model must select get_stock_prices with 3mo.
# The explicit exclusion is checked before execution; a financial-statement call
# is rejected and retried within the same bounded fallback budget
price_only_run = team.execute_project(
    "MSFT",
    "Retrieve the last three months of daily stock prices only. "
    "Do not retrieve company financial statements.",
    price_period="3mo",
    excluded_tools=["get_company_financials"],
)

items = price_only_run["tool_results"]
passed = (
    price_only_run["status"] == "success"
    and len(items) == 1
    and items[0]["tool"] == "get_stock_prices"
    and items[0]["arguments"] == {"ticker": "MSFT", "period": "3mo"}
)
print("Dynamic tool selection test:", "PASS" if passed else "FAIL")
if not passed:
    raise AssertionError("The requested price-only research did not succeed.")

display(team.show_team_status())

Coordinator: attempt 1/3 — nvidia/nemotron-3-super-120b-a12b:free
Coordinator: accepted response from nvidia/nemotron-3-super-120b-a12b:free
Coordinator → Researcher: Retrieve the last three months of daily stock prices only. Do not retrieve company financial statements.
Researcher: attempt 1/3 — nvidia/nemotron-3-super-120b-a12b:free
Researcher: accepted response from nvidia/nemotron-3-super-120b-a12b:free
Researcher: get_stock_prices({'ticker': 'MSFT', 'period': '3mo'}) → success
Researcher → Coordinator: Retrieve the last three months of daily stock prices only. Do not retrieve company financial statements.
Dynamic tool selection test: PASS


,agent,completed_tasks_this_session
0,Coordinator,2
1,Researcher,2


## **<font color="green">Handoff to Robert here - DELETE WHEN DONE</font>**

Use research_run as the input to the next stage. It includes the original question, plan, tool results, model IDs, attempts and delegation log. Check its status before analysis; partial or error means some evidence is unavailable. save_handoff(research_run) can export the packet as JSON if needed.

Some helpful hints for the previously added code:

Base Agent: reuse as it is, no change needed.

ResearcherAgent: keep its Yahoo retrieval; extend it with financial analysis.

New NewsResearcherAgent: add news retrieval and the prompt chain.

Coordinator: modify it to route tasks to the appropriate researcher and collect results.

MultiAgentTeam: register the new news agent if you decide to .

---
# 5. Financial News Processing: Prompt Chaining

Implement ingest news → preprocess → classify → extract → summarize.

---
# 6. Content Routing and Specialist Analysis

Add news, earnings and market-analysis methods to ResearcherAgent or create a new NewsReasearcherAgent. Add the routing decision to Coordinator, then delegate the selected task with send_to. Keep the evidence and add analysis results to the shared packet.

## **<font color="green"> Handoff to Erika here - DELETE WHEN DONE </font>**

---
# 7. Self-Reflection and Evaluator–Optimizer Workflow

Add EvaluatorAgent as the third or fourth agent dependig on whether Robert utilized the ResearcherAgent or created a new NewsResearcherAgent. Evaluate analyses and send specific feedback to ResearcherAgent and/or other agents created for a bounded number of revisions.

---
# 8. Memory and Learning Across Runs

Persist and retrieve lessons that improve subsequent analyses. The current Agent.memory tracks tasks during the session; persistent learning is not implemented yet.

---
# 9. Final Research Report and Visualizations

Add final synthesis to the coordinator and create evidence-based visualizations. No separate writer agent is needed.

## **<font color="green"> Work Altogether From Here - DELETE WHEN DONE </font>**

---
# 10. End-to-End Demonstration and Testing

The current AAPL and MSFT demonstrations appear in Section 4. Add the complete team workflow and its genuine outputs here once the remaining agents are implemented.

---
# 11. Conclusions, Limitations and Future Improvements


---
# 12. References